# Fine-tune an 8B model on 4 GB of VRAM — run it yourself

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MakazhanAlpamys/Soup/blob/main/notebooks/proof-4gb.ipynb)

[Soup](https://github.com/MakazhanAlpamys/Soup) trains a model whose weights do not fit
in your GPU. The frozen base stays in host RAM and is streamed to the GPU one decoder
layer at a time, so peak VRAM is bounded by **one layer** instead of by the model.

This notebook does not ask you to believe that. It caps this process to **4 GB** on
Colab's free T4 and then measures what actually happens.

| Section | What it proves | Time |
|---|---|---|
| 1–3 | The cap is real, and this GPU has no bf16 | ~2 min |
| 4 | A streamed model and a normal one produce **bit-identical** logits | ~3 min |
| 5 | Llama-3.1-8B trains with a measured peak under 4 GB | ~20 min |

Sections 1–4 are the argument. Section 5 is the headline and is optional.

**Runtime → Change runtime type → T4 GPU** before you start.


## 1. Install

**From git, not PyPI, and the reason is the point of section 2.** The fix that makes
this pick the right precision on a T4
([#385](https://github.com/MakazhanAlpamys/Soup/issues/385),
[#387](https://github.com/MakazhanAlpamys/Soup/issues/387)) is on `main` and is not in a
release yet, so on the published 0.73.0 the next cell raises `ImportError`. Switch this
line back to `soup-cli[train]` once the next version ships.

The `torchao` line is not incidental either. Colab preinstalls **torchao 0.10.0**, and
`peft` does not merely decline to use a version it considers too old — it *raises*
`ImportError` from `is_torchao_available()`, several frames inside `get_peft_model`.
Nothing here needs torchao, so it is removed rather than upgraded (upgrading risks
pulling a wheel built against a different torch).


In [ ]:
%pip uninstall -q -y torchao
%pip install -q "soup-cli[train] @ git+https://github.com/MakazhanAlpamys/Soup.git"

import importlib.util

import soup_cli
import soup_cli.utils.gpu

print("soup", soup_cli.__version__)
print("pre-Ampere fix present:", hasattr(soup_cli.utils.gpu, "bf16_fp16_flags"))
print("torchao gone (peft raises on an old one):",
      importlib.util.find_spec("torchao") is None)


## 2. What card did we get, and does it have bf16?

Colab's free tier is a **T4** — Turing, sm_75. bf16 hardware arrived with Ampere, so a
T4 has none.

**Read the two lines below carefully, because they disagree, and the disagreement is
the point.** `torch.cuda.is_bf16_supported()` defaults to `including_emulation=True`:
when the compute-capability check fails it falls through to *constructing* a bf16
tensor, which software emulation satisfies. So a T4 answers **True** to the question
everyone asks, and False only to `is_bf16_supported(including_emulation=False)`.

Soup asked the permissive question and therefore handed bf16 to a card with no bf16
units. The first version of this fix asked it too, and was a no-op on exactly the
hardware it was written for — caught by running this notebook on a real T4, not before
([#385](https://github.com/MakazhanAlpamys/Soup/issues/385),
[#387](https://github.com/MakazhanAlpamys/Soup/issues/387)).


In [ ]:
import torch

from soup_cli.utils.gpu import bf16_fp16_flags

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU."

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
total = torch.cuda.get_device_properties(0).total_memory

print(f"GPU                  {name}  (sm_{major}{minor})")
print(f"VRAM                 {total / 1e9:.1f} GB")
print(f"bf16, incl. emulation {torch.cuda.is_bf16_supported()}")
print(f"bf16 IN HARDWARE      "
      f"{torch.cuda.is_bf16_supported(including_emulation=False)}")

bf16, fp16 = bf16_fp16_flags("cuda")
print(f"Soup will train in   {'bf16' if bf16 else 'fp16' if fp16 else 'fp32'}")


## 3. Cap this process to 4 GB

`set_per_process_memory_fraction` caps PyTorch's allocator. Everything after this cell
runs as if the card were a 4 GB laptop GPU — an allocation past the cap raises, exactly
as it would on the real thing.

**One honest caveat.** The cap is enforced by the allocator, not by the driver, so
`torch.cuda.mem_get_info()` keeps reporting the *whole* card. Soup's pre-flight reads
that, so the "free VRAM" line it prints later belongs to the host card and not to this
capped process — it will happily allow a configuration that the allocator then refuses.
That is [#347](https://github.com/MakazhanAlpamys/Soup/issues/347), it is open, and it
does not affect anything measured here: the proof below is the **peak VRAM torch
actually reports**, not what the pre-flight predicted.


In [ ]:
BUDGET_BYTES = 4 * 1000**3  # 4 GB, the card this method was developed on

fraction = BUDGET_BYTES / total
torch.cuda.set_per_process_memory_fraction(fraction)
print(f"capped at {BUDGET_BYTES / 1e9:.2f} GB  (fraction {fraction:.3f} of this card)")

# Prove the cap bites: ask for 15% more than the budget and expect a refusal.
try:
    _ = torch.empty(int(BUDGET_BYTES * 1.15), dtype=torch.uint8, device="cuda")
    print("WARNING: the allocation succeeded — the cap is NOT in force")
except RuntimeError as exc:
    print("refused, as it should be:", str(exc).splitlines()[0][:90])


## 4. The claim that matters: streamed == resident, bit for bit

A streaming bug is silent. If the base were substituted wrongly, or the autograd path
severed, the loss would still fall — the upper layers keep learning — and you would ship
a damaged model without an error anywhere.

So the check is not "does it train". It is: **the same weights, through the same kernels,
must produce the same numbers.** Below, one model is streamed layer-by-layer and the
other is an ordinary resident model, both carrying identical adapter weights.
`torch.equal` is exact equality, not a tolerance.

This runs on a small model because the reference has to fit in memory next to the
streamed copy — that is the whole reason the headline size cannot be checked this way.


In [ ]:
import tempfile
from pathlib import Path

from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM

from soup_cli.utils.layer_shard import shard_checkpoint
from soup_cli.utils.layer_stream import resolve_stream_dtype
from soup_cli.utils.layer_stream_runtime import build_streamed_model
from soup_cli.utils.spectrum_scan import resolve_model_weights

MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
DTYPE = resolve_stream_dtype("cuda")  # fp16 on a T4, bf16 on an Ampere card
LORA = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.0, bias="none",
    target_modules=["q_proj", "v_proj"], task_type=TaskType.CAUSAL_LM,
)

workdir = Path(tempfile.mkdtemp())
weights = resolve_model_weights(MODEL)  # downloads on first use
index = shard_checkpoint(weights, str(workdir / "shards"), dtype=DTYPE, arch="llama")
streamed, runtime = build_streamed_model(
    model_id=weights, shard_dir=str(workdir / "shards"), index=index,
    lora_config=LORA, device="cuda", dtype=DTYPE, buffers=2, pin=True, seed=0,
)
print(f"streamed: {index.n_layers} layers, dtype={DTYPE}")


In [ ]:
# PEFT initialises lora_B to zero, so an untrained adapter contributes NOTHING and any
# comparison would silently be about the base model alone. Make it load-bearing first.
gen = torch.Generator().manual_seed(7)
with torch.no_grad():
    for pname, param in streamed.named_parameters():
        if "lora_B" in pname:
            param.copy_(torch.randn(param.shape, generator=gen).to(param.device, param.dtype))

resident = AutoModelForCausalLM.from_pretrained(
    MODEL, dtype=getattr(torch, DTYPE), device_map={"": "cuda"}
)
resident = get_peft_model(resident, LORA)

# Copy the adapter across. The streamed wrapper inserts an '.inner.' segment in its keys.
src = {k.replace(".inner.", "."): v for k, v in streamed.state_dict().items() if "lora_" in k}
dst = {k.replace(".inner.", "."): v for k, v in resident.state_dict().items() if "lora_" in k}
assert src and set(src) == set(dst)
with torch.no_grad():
    for key, val in src.items():
        dst[key].copy_(val.to(dst[key].dtype))

ids = torch.randint(0, 4096, (1, 32), device="cuda")
with torch.no_grad():
    a = streamed(input_ids=ids).logits
    b = resident(input_ids=ids).logits

print("max |streamed - resident| =", (a.float() - b.float()).abs().max().item())
print("torch.equal              =", torch.equal(a, b))
assert torch.equal(a, b), "NOT bit-exact — please open an issue with this output"
print("\nBit-exact. The streamed model is the same model.")


In [ ]:
import gc

# Free the reference before the headline run.
runtime.close()
del streamed, resident, a, b
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print("reset")


## 5. The headline: Llama-3.1-8B, trained under the 4 GB cap

8B parameters. In NF4 the weights alone are about **4.5 GB** — more than the budget this
process is allowed, before activations, gradients or the optimizer. It trains anyway,
because at any moment only a couple of decoder layers are resident.

Expect ~20 minutes, most of it the download. The number to watch is the **peak VRAM**
at the end — not the loss. This is a handful of steps on 32 toy rows; over that distance
the loss can go up as easily as down, and it would prove nothing either way. What is
being demonstrated is that the run *happens at all* inside the budget.


In [ ]:
import json

# Varied on both sides on purpose: 32 copies of one answer drive the loss to 0.000
# immediately and the printed curve stops meaning anything.
TOPICS = [
    ("streaming", "Only a couple of decoder layers are resident at any moment."),
    ("NF4", "Four-bit weights make the host-side store about four times smaller."),
    ("LoRA", "The base is frozen, so it is read and never written."),
    ("VRAM", "Peak memory is bounded by one layer instead of by the model."),
]
rows = [
    {"messages": [
        {"role": "user", "content": f"Question {i}: tell me about {topic}."},
        {"role": "assistant", "content": answer},
    ]}
    for i in range(8)
    for topic, answer in TOPICS
]
Path("train.jsonl").write_text("\n".join(json.dumps(r) for r in rows), encoding="utf-8")

config = """
base: NousResearch/Meta-Llama-3.1-8B-Instruct
task: sft
data:
  train: train.jsonl
  max_length: 256
training:
  epochs: 1
  batch_size: 1
  lr: 0.0002
  logging_steps: 1        # short run — without this nothing gets logged
  quantization: 4bit      # NF4 — the store is ~4x smaller than bf16
  stream_layers: true     # the feature
  stream_buffers: 2
  lora:
    r: 8
    alpha: 16
output: ./out-8b
"""
Path("soup.yaml").write_text(config, encoding="utf-8")
print(config)


The cell below runs the trainer **in this process** rather than shelling out to
`soup train`. That is deliberate and it is the only reason: `max_memory_allocated()`
reports the peak of *the process that calls it*, so a subprocess would train fine and
leave us measuring nothing. It is the same code path the CLI runs on the same
`soup.yaml` above — the CLI adds argument parsing and the pre-flight panel, neither of
which changes what the GPU does.


In [ ]:
from soup_cli.config.loader import load_config_from_string
from soup_cli.data.loader import load_dataset
from soup_cli.trainer.sft import SFTTrainerWrapper

cfg = load_config_from_string(config)
dataset = load_dataset(cfg.data)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

wrapper = SFTTrainerWrapper(cfg)
wrapper.setup(dataset)   # downloads, shards to NF4, builds the streamed model
result = wrapper.train()

print(f"\nsteps: {result['total_steps']}  loss: {result['initial_loss']:.3f}"
      f" -> {result['final_loss']:.3f}")


In [ ]:
peak = torch.cuda.max_memory_allocated()
print(f"peak VRAM allocated by this process: {peak / 1e9:.2f} GB")
print(f"budget this process was capped to:   {BUDGET_BYTES / 1e9:.2f} GB")
print("model weights in NF4, for scale:     ~4.5 GB")

adapter = Path("out-8b/adapter_model.safetensors")
print(f"\nadapter written: {adapter.exists()}")
if adapter.exists():
    from safetensors.torch import load_file

    tensors = load_file(str(adapter))
    live = sum(1 for v in tensors.values() if v.abs().max().item() > 0)
    print(f"adapter tensors: {len(tensors)}, non-zero: {live}")


## What this proved, and what it did not

**Proved, on your hardware:**

- A streamed model returns **bit-identical** logits to an ordinary one (§4).
- An 8B model trained with a measured peak below a cap smaller than its own weights (§5).
- Both on a GPU with **no bf16** — the case that was broken until recently.

**Not proved:**

- *Backward* exactness at this size. §4 compares the forward. Gradient exactness is
  verified up to 14B against resident references on hardware that can hold them, and
  a defect **above** that size was found, named upstream and repaired — see
  [`benchmarks/`](https://github.com/MakazhanAlpamys/Soup/tree/main/benchmarks).
- Speed. A T4 under an artificial cap is not a throughput benchmark, and this notebook
  deliberately does not quote tok/s.

Layer streaming is **BETA** and opt-in (`stream_layers: true`).

**If any assertion above failed, that is worth an issue** — with the cell output. A
reproduction on hardware we do not own is more useful to this project than a star.

The method, the correctness protocol and every measurement:
[10.5281/zenodo.21771064](https://doi.org/10.5281/zenodo.21771064) ·
[measurement records](https://github.com/MakazhanAlpamys/Soup/tree/main/benchmarks)
